In [9]:
from pathlib import Path
import shutil

root = Path("/kaggle/input")
for p in root.rglob("*"):
    if p.is_dir() and ("MoNuSeg" in p.name or "Images" in p.name or "Annotations" in p.name):
        print(p)

/kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSeg 2018 Training Data
/kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSegTestData
/kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSeg 2018 Training Data/MoNuSeg 2018 Training Data
/kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSegTestData/MoNuSegTestData
/kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSeg 2018 Training Data/MoNuSeg 2018 Training Data/Tissue Images
/kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSeg 2018 Training Data/MoNuSeg 2018 Training Data/Annotations


In [10]:
src_root = Path("/kaggle/input")

image_dirs = []
xml_dirs = []

for p in src_root.rglob("*"):
    if p.is_dir():
        names = [f.suffix.lower() for f in p.iterdir() if f.is_file()]
        if ".tif" in names:
            image_dirs.append(p)
        if ".xml" in names:
            xml_dirs.append(p)

print("Image dirs:")
for d in image_dirs: print(" ", d)

print("\nXML dirs:")
for d in xml_dirs: print(" ", d)


# create expected structure
dst = Path("/kaggle/working/monuseg_prepared")
(dst / "training" / "images").mkdir(parents=True, exist_ok=True)
(dst / "training" / "labels").mkdir(parents=True, exist_ok=True)
(dst / "testing" / "images").mkdir(parents=True, exist_ok=True)
(dst / "testing" / "labels").mkdir(parents=True, exist_ok=True)

# copy by filename pattern (training vs test is in the names)
for d in image_dirs:
    for f in d.glob("*.tif"):
        if "Test" in str(d) or "Test" in f.name:
            shutil.copy(f, dst / "testing" / "images" / f.name)
        else:
            shutil.copy(f, dst / "training" / "images" / f.name)

for d in xml_dirs:
    for f in d.glob("*.xml"):
        if "Test" in str(d) or "Test" in f.name:
            shutil.copy(f, dst / "testing" / "labels" / f.name)
        else:
            shutil.copy(f, dst / "training" / "labels" / f.name)

print("Copy finished.")

Image dirs:
  /kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSegTestData/MoNuSegTestData
  /kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSeg 2018 Training Data/MoNuSeg 2018 Training Data/Tissue Images

XML dirs:
  /kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSegTestData/MoNuSegTestData
  /kaggle/input/datasets/kangaonkaggle/monuseg-data/MoNuSeg 2018 Training Data/MoNuSeg 2018 Training Data/Annotations
Copy finished.


In [11]:
root = Path("/kaggle/working/monuseg_prepared")

for p in [
    root / "training" / "images",
    root / "training" / "labels",
    root / "testing" / "images",
    root / "testing" / "labels",
]:
    print("\n", p)
    if p.exists():
        files = list(p.iterdir())
        print("count:", len(files))
        for f in files[:5]:
            print(" ", f.name)
    else:
        print("MISSING")


 /kaggle/working/monuseg_prepared/training/images
count: 37
  TCGA-HE-7128-01Z-00-DX1.tif
  TCGA-B0-5711-01Z-00-DX1.tif
  TCGA-49-4488-01Z-00-DX1.tif
  TCGA-E2-A14V-01Z-00-DX1.tif
  TCGA-RD-A8N9-01A-01-TS1.tif

 /kaggle/working/monuseg_prepared/training/labels
count: 37
  TCGA-RD-A8N9-01A-01-TS1.xml
  TCGA-18-5592-01Z-00-DX1.xml
  TCGA-FG-A87N-01Z-00-DX1.xml
  TCGA-A7-A13F-01Z-00-DX1.xml
  TCGA-B0-5710-01Z-00-DX1.xml

 /kaggle/working/monuseg_prepared/testing/images
count: 14
  TCGA-2Z-A9J9-01A-01-TS1.tif
  TCGA-AC-A2FO-01A-01-TS1.tif
  TCGA-HT-8564-01Z-00-DX1.tif
  TCGA-FG-A4MU-01B-01-TS1.tif
  TCGA-A6-6782-01A-01-BS1.tif

 /kaggle/working/monuseg_prepared/testing/labels
count: 14
  TCGA-AC-A2FO-01A-01-TS1.xml
  TCGA-44-2665-01B-06-BS6.xml
  TCGA-FG-A4MU-01B-01-TS1.xml
  TCGA-EJ-A46H-01A-03-TSC.xml
  TCGA-CU-A0YN-01A-02-BSB.xml


In [ ]:
# -*- coding: utf-8 -*-
# Prepare MoNuSeg Dataset By converting and resorting files
#
# @ Fabian Hörst, fabian.hoerst@uk-essen.de
# Institute for Artifical Intelligence in Medicine,
# University Medicine Essen

from PIL import Image
import xml.etree.ElementTree as ET
from skimage import draw
import numpy as np
from pathlib import Path
from typing import Union
import argparse


def convert_monuseg(
    input_path: Union[Path, str], output_path: Union[Path, str]
) -> None:
    """Convert the MoNuSeg dataset to a new format (no resize, tiff to png and xml to npy)

    Args:
        input_path (Union[Path, str]): Input dataset
        output_path (Union[Path, str]): Output path
    """
    input_path = Path(input_path)
    output_path = Path(output_path)
    output_path.mkdir(exist_ok=True, parents=True)

    # testing and training
    parts = ["testing", "training"]
    for part in parts:
        print(f"Prepare: {part}")
        input_path_part = input_path / part
        output_path_part = output_path / part
        output_path_part.mkdir(exist_ok=True, parents=True)
        (output_path_part / "images").mkdir(exist_ok=True, parents=True)
        (output_path_part / "labels").mkdir(exist_ok=True, parents=True)

        # images
        images = [f for f in sorted((input_path_part / "images").glob("*.tif"))]
        for img_path in images:
            loaded_image = Image.open(img_path)
            resized = loaded_image.resize(
                (1000, 1000), resample=Image.Resampling.LANCZOS
            )
            new_img_path = output_path_part / "images" / f"{img_path.stem}.png"
            resized.save(new_img_path)
        # masks
        annotations = [f for f in sorted((input_path_part / "labels").glob("*.xml"))]
        for annot_path in annotations:
            binary_mask = np.zeros((1000, 1000), dtype=np.int32)

            # extract xml file
            tree = ET.parse(annot_path)
            root = tree.getroot()
            child = root[0]

            for x in child:
                r = x.tag
                if r == "Regions":
                    element_idx = 1
                    for y in x:
                        y_tag = y.tag

                        if y_tag == "Region":
                            regions = []
                            vertices = y[1]
                            coords = np.zeros((len(vertices), 2))
                            for i, vertex in enumerate(vertices):
                                coords[i][0] = vertex.attrib["X"]
                                coords[i][1] = vertex.attrib["Y"]
                            regions.append(coords)
                            vertex_row_coords = regions[0][:, 0]
                            vertex_col_coords = regions[0][:, 1]
                            fill_row_coords, fill_col_coords = draw.polygon(
                                vertex_col_coords, vertex_row_coords, binary_mask.shape
                            )
                            binary_mask[fill_row_coords, fill_col_coords] = element_idx

                            element_idx = element_idx + 1
            inst_image = Image.fromarray(binary_mask)
            resized_mask = np.array(
                inst_image.resize((1000, 1000), resample=Image.Resampling.NEAREST)
            )
            new_mask_path = output_path_part / "labels" / f"{annot_path.stem}.npy"
            np.save(new_mask_path, resized_mask)
    print("Finished")


parser = argparse.ArgumentParser(
    formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    description="Convert the MoNuSeg dataset",
)
parser.add_argument(
    "--input_path",
    type=str,
    help="Input path of the original MoNuSeg dataset",
    required=True,
)
parser.add_argument(
    "--output_path",
    type=str,
    help="Output path to store the processed MoNuSeg dataset",
    required=True,
)

convert_monuseg(
    "/kaggle/working/monuseg_prepared",
    "/kaggle/working/monuseg_converted",
)

Prepare: testing
